In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

In [117]:

df=pd.read_csv("/content/Gaming_Academic_Performance.csv")
df.head()



,student_id,age,gender,gaming_hours,study_hours,sleep_hours,attendance,gaming_genre,social_activity,device_usage,reaction_time_ms,addiction_score,stress_level,grades
0,1,22,Male,7.23,8.78,6.96,91.44,FPS,3.25,9.36,235.84,14.69,Low,86.459555
1,2,19,Male,0.07,8.72,7.63,63.63,Casual,1.02,3.21,328.71,2.47,Medium,98.230000
2,3,23,Female,1.73,9.56,4.40,83.26,Casual,3.46,5.56,313.61,4.73,High,90.560000
3,4,20,Female,6.62,1.68,7.83,75.04,RPG,1.46,11.78,241.84,14.54,Low,32.670000
4,5,22,Female,5.36,5.83,5.55,65.57,FPS,1.01,8.23,249.31,12.48,Low,58.710000


In [119]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        8000 non-null   int64  
 1   age               8000 non-null   int64  
 2   gender            8000 non-null   object 
 3   gaming_hours      8000 non-null   float64
 4   study_hours       8000 non-null   float64
 5   sleep_hours       8000 non-null   float64
 6   attendance        8000 non-null   float64
 7   gaming_genre      8000 non-null   object 
 8   social_activity   8000 non-null   float64
 9   device_usage      8000 non-null   float64
 10  reaction_time_ms  8000 non-null   float64
 11  addiction_score   8000 non-null   float64
 12  stress_level      8000 non-null   object 
 13  grades            8000 non-null   float64
dtypes: float64(9), int64(2), object(3)
memory usage: 875.1+ KB


# Train-Test Split

In [120]:
X_train, X_test, y_train, y_test = train_test_split (df.iloc[:,0:13], df.iloc[:,-1], test_size=0.2, random_state=42)

In [121]:
X_train

,student_id,age,gender,gaming_hours,study_hours,sleep_hours,attendance,gaming_genre,social_activity,device_usage,reaction_time_ms,addiction_score,stress_level
1467,1468,24,Male,4.04,1.38,6.93,76.68,FPS,2.55,5.57,270.08,9.41,Medium
5768,5769,21,Female,2.52,4.59,4.02,91.11,FPS,1.47,7.86,290.56,6.94,Medium
5714,5715,23,Female,7.73,6.08,8.65,91.49,Casual,2.32,9.57,238.54,15.24,Low
1578,1579,24,Male,5.74,3.45,6.11,65.12,FPS,4.84,10.53,260.72,10.85,Low
6958,6959,20,Female,2.43,2.06,6.54,60.48,Casual,1.48,5.82,299.33,9.10,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5226,5227,19,Female,4.41,3.75,5.31,91.84,Casual,4.11,6.90,266.37,11.42,Medium
5390,5391,24,Male,0.52,2.02,6.89,96.20,RPG,3.13,5.45,307.09,1.52,Medium
860,861,16,Male,7.47,3.60,4.51,73.10,FPS,2.14,13.06,244.90,17.64,Low
7603,7604,19,Male,0.64,8.02,8.01,65.14,FPS,1.30,6.35,307.95,0.90,Medium


# Pipeline

In [122]:
stage1= ColumnTransformer([
    ('ohe_gen_gamgen',OneHotEncoder(handle_unknown='ignore'), [2,7]),
    ('oe_stress_level',OrdinalEncoder(categories=[['Low', 'Medium', 'High']]), [12])
],remainder='passthrough')

In [123]:
stage2= ColumnTransformer([
    ('scale',StandardScaler(),[3,4,5,6,8,9,10,11])
],remainder='passthrough')

In [124]:
stage3=LinearRegression()

In [125]:
pipe = Pipeline([
    ('stage1',stage1),
    ('stage2',stage2),
    ('stage3',LinearRegression())
])

# Training

In [126]:
pipe.fit(X_train,y_train)
pipe

Pipeline(steps=[('stage1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_gen_gamgen',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  [2, 7]),
                                                 ('oe_stress_level',
                                                  OrdinalEncoder(categories=[['Low',
                                                                              'Medium',
                                                                              'High']]),
                                                  [12])])),
                ('stage2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scale', StandardScaler(),
                                                  [3, 4, 5, 6, 8, 9, 10,
                                                   11])])),
                ('stage3', LinearRegression())])

In [127]:
from sklearn import set_config
set_config(display='diagram')

# Prediction

In [128]:
y_pred=pipe.predict(X_test)

In [129]:
y_pred

array([45.52465491, 73.94417367, 49.19657952, ..., 53.97047458,
       97.40160752, 85.52456173])

In [130]:
r2_score(y_test,y_pred)

0.9031093158737746

# Pickle File

In [131]:
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))